# Generative Models and RAG

In the previous notebook we opened up the Transformer and fine-tuned BERT for a classification task and updated the model's weights with gradient descent. 

Today we do something different: we use large, already-trained generative models purely through **prompting**, with no training step at all. 

We will compare running a model **locally** (via [Ollama](https://ollama.com/)) against calling **frontier model APIs** (Mistral, OpenAI), and then use these same building blocks to implement a **Retrieval-Augmented Generation (RAG)** system -- first from scratch, then via the `rag_app/` Chainlit application already in this repository.

Day 11 (today, hours 1-2) covers local vs. frontier generative models in practice. Hour 3 is Q&A. Day 12 covers RAG: first a from-scratch implementation, then a walkthrough of `rag_app/`, then exercises.

## From fine-tuning to prompting

Notebook 4 fine-tuned a pre-trained BERT encoder on a classification task: we added a small head, ran the training loop, and updated weights. The models we use today are used exclusively via **in-context learning** -- we write a prompt, the model reads it and generates a continuation, and no parameter is ever updated. All the "learning" happens at pre-training time (and, for chat models, at a subsequent instruction-tuning / RLHF stage) that we don't have access to or control over.

A few concepts that matter once you move from fine-tuning to prompting a generative model:

* **Decoder-only architecture.** Unlike the encoder-decoder Transformer shown in Notebook 4 (used for translation-like tasks), most modern chat/generative models (GPT, Mistral, Llama) are **decoder-only**: a single stack of Transformer blocks that only ever attends to the past, trained to predict the next token.
* **Autoregressive sampling.** Generation happens one token at a time: the model outputs a probability distribution over the vocabulary for the next token, one token is sampled (or picked greedily), appended to the input, and the process repeats.
* **Temperature.** A parameter that reshapes that probability distribution before sampling. `temperature=0` is (close to) deterministic/greedy decoding; higher temperatures flatten the distribution, giving more varied (and less predictable) output.
* **Context window.** The maximum number of tokens (input + output combined) the model can process in one call. Everything you send -- system prompt, conversation history, retrieved documents -- counts against this budget.
* **Tokens map directly to cost.** Commercial APIs bill per input and per output token. Local models cost only compute/electricity. This matters a lot for anything you plan to run at the scale of thousands of documents (see the discussion later in this notebook).

## Setup

**API keys.** The Mistral and OpenAI SDKs, and the Ollama client, are already declared in `requirements.txt`. You need both a `MISTRAL_API_KEY` (for the frontier-model examples in this notebook) and an `OPENAI_API_KEY` (for `rag_app/`, which defaults to OpenAI). The convention used across this course (see `rag_app/app.py`) is to keep secrets in a local `.env` file and load them with `python-dotenv`:

```
# .env  (copy from .env.template in the repo root -- never commit the real .env)
MISTRAL_API_KEY=...
OPENAI_API_KEY=sk-...
```

Copy `.env.template` to `.env` and fill in your own keys: `cp .env.template .env`.

**Never commit `.env` or hardcode API keys in a notebook.** Add `.env` to `.gitignore` if it isn't already there. Treat a leaked key the same as a leaked password -- revoke it immediately from the provider's dashboard.

**Rate limits.** Free/trial-tier Mistral keys allow only about one request per second and cap monthly token usage. Every paid call in this notebook goes through a small retry-with-backoff helper (defined below), so an occasional "Rate limited, retrying in Ns..." message during a run is expected behavior, not a bug.

**Local models.** Before class, install [Ollama](https://ollama.com/) and pull two small models that fit comfortably on a laptop:

```
ollama pull llama3.2:3b
ollama pull nomic-embed-text
```

The cell below just checks that the `ollama` CLI is installed and prints its version -- it is expected to fail (harmlessly) if Ollama isn't installed on this machine, or in an environment where you're only running the frontier-API cells.

In [ ]:
!ollama --version || echo "Ollama not found -- that's fine if you're only running the API cells below."

## Local: Ollama

Ollama runs open-weight models (Llama, Mistral, Gemma, ...) locally and exposes them through a small Python client and a local HTTP server. Everything in this section requires a **running Ollama daemon** (`ollama serve`, usually started automatically once you launch the app) -- like the FashionMNIST download in Notebook 3, these cells assume a local resource that may not be available on every machine, so treat a connection error here as expected rather than a bug in the code.

In [ ]:
import ollama

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": "In one sentence, what is Retrieval-Augmented Generation?"}
    ],
)
print(response["message"]["content"])

Streaming works by setting `stream=True` and iterating over the response -- useful for anything user-facing, since the reader sees tokens as they're generated instead of waiting for the full response.

In [ ]:
stream = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": "List three uses of machine learning in the humanities."}],
    stream=True,
)

for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)

The `options` dict controls sampling parameters such as `temperature`, and a `system` message sets the model's persona/instructions -- exactly the same idea as the `system_prompt` used in `rag_app/app.py`.

In [ ]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": "You are a terse research assistant for digital humanities. Answer in at most two sentences."},
        {"role": "user", "content": "Why might a historian be skeptical of OCR'd text?"},
    ],
    options={"temperature": 0.2},
)
print(response["message"]["content"])

Finally, `ollama.embed` produces vector embeddings locally, with no API cost -- this is the piece we reuse on Day 12 to build a RAG pipeline entirely offline.

In [ ]:
embed_response = ollama.embed(model="nomic-embed-text", input="Retrieval-Augmented Generation combines a retriever with a generator.")
embedding = embed_response["embeddings"][0]
print("Embedding dimensionality:", len(embedding))
print(embedding[:8], "...")

## Frontier: the Mistral API

Commercial APIs give you access to much larger, more capable models without having to host anything yourself -- at a per-token cost, and with your data leaving your machine. The `mistralai` Python package talks to Mistral models the same way `ollama` talks to local ones.

In [ ]:
import os
import time
from dotenv import load_dotenv
from mistralai import Mistral

load_dotenv()  # reads MISTRAL_API_KEY (and OPENAI_API_KEY) from a local .env file

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
MISTRAL_MODEL = "mistral-small-latest"


def with_backoff(call, max_retries=5):
    """Run `call()`, retrying with exponential backoff on Mistral rate limits (HTTP 429).

    Free-tier Mistral keys allow only about one request per second, so every paid call
    in this notebook goes through this helper rather than calling the SDK directly.
    """
    for attempt in range(max_retries):
        try:
            return call()
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


response = with_backoff(lambda: client.chat.complete(
    model=MISTRAL_MODEL,
    max_tokens=256,
    messages=[
        {"role": "system", "content": "You are a terse research assistant for digital humanities."},
        {"role": "user", "content": "In one sentence, what is Retrieval-Augmented Generation?"},
    ],
))
print(response.choices[0].message.content)

As with Ollama, streaming avoids waiting for the whole response before showing anything to the user:

In [ ]:
stream_response = with_backoff(lambda: client.chat.stream(
    model=MISTRAL_MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "List three uses of machine learning in the humanities."}],
))

for chunk in stream_response:
    delta = chunk.data.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

Every response carries a `usage` field with the exact number of input and output tokens consumed (`prompt_tokens` and `completion_tokens`). Multiplying by a published price-per-million-tokens gives an estimated cost -- update `price_per_million_input` / `price_per_million_output` with the current numbers from the provider's pricing page, since they change over time and we don't want to hardcode a value here that may go stale.

In [ ]:
price_per_million_input = 0.10   # USD -- update from the current Mistral pricing page (this used Mistral Small pricing at the time of writing)
price_per_million_output = 0.30  # USD -- update from the current Mistral pricing page

usage = response.usage
input_cost = usage.prompt_tokens / 1_000_000 * price_per_million_input
output_cost = usage.completion_tokens / 1_000_000 * price_per_million_output

print(f"Input tokens: {usage.prompt_tokens}, output tokens: {usage.completion_tokens}")
print(f"Estimated cost: ${input_cost + output_cost:.6f}")

Two further capabilities are worth knowing exist, even though we won't use them today: **tool use** (letting the model call functions you define -- e.g. a search function, a calculator, or a database query) and **prompt caching** (reusing large, repeated chunks of context across calls at a fraction of the cost). Both are central to building more complex agentic applications, but out of scope for this notebook.

## Head-to-head on a humanities task

Let's compare the local model and the Mistral API on a realistic task from this course: extracting structured metadata from OCR'd 19th-century book text. We reuse the British Library book-text dataset from `data/bl_books/sample_tidy/` and ask both models to extract a title, place, date, and any persons mentioned, as JSON.

In [ ]:
import pandas as pd

df_book = pd.read_csv("data/bl_books/sample_tidy/df_book.csv")
df_text = pd.read_csv("data/bl_books/sample_tidy/df_book_text.csv")

sample = df_text.head(3).copy()
sample["fulltext"] = sample["fulltext"].str.slice(0, 1500)  # keep the prompt short -- just the opening of each book
sample.head()

In [ ]:
extraction_prompt_template = '''The following text is OCR'd from the opening pages of a 19th-century book. \
Extract the following fields as a single JSON object with keys "title", "place", "date", "persons" \
(a list of personal names mentioned). If a field cannot be determined, use null.

TEXT:
"""{text}"""
'''

In [ ]:
import json

def extract_with_ollama(text, model="llama3.2:3b"):
    prompt = extraction_prompt_template.replace("{text}", text)
    start = time.time()
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    latency = time.time() - start
    return response["message"]["content"], latency, None  # Ollama is free/local -- no token cost to report


def extract_with_mistral(text, model=MISTRAL_MODEL):
    prompt = extraction_prompt_template.replace("{text}", text)
    start = time.time()
    response = with_backoff(lambda: client.chat.complete(
        model=model,
        max_tokens=512,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    ))
    latency = time.time() - start
    usage = response.usage
    cost = (usage.prompt_tokens / 1_000_000 * price_per_million_input
            + usage.completion_tokens / 1_000_000 * price_per_million_output)
    return response.choices[0].message.content, latency, cost

In [ ]:
results = []
for _, row in sample.iterrows():
    ollama_output, ollama_latency, _ = extract_with_ollama(row["fulltext"])
    mistral_output, mistral_latency, mistral_cost = extract_with_mistral(row["fulltext"])
    results.append({
        "fulltext_filename": row["fulltext_filename"],
        "ollama_latency_s": round(ollama_latency, 2),
        "ollama_output": ollama_output,
        "mistral_latency_s": round(mistral_latency, 2),
        "mistral_cost_usd": round(mistral_cost, 6),
        "mistral_output": mistral_output,
        "quality_notes": "",  # fill in manually after inspecting both outputs
    })
    time.sleep(1)  # stay under Mistral's free-tier requests-per-second limit

comparison = pd.DataFrame(results)
comparison[["fulltext_filename", "ollama_latency_s", "mistral_latency_s", "mistral_cost_usd", "quality_notes"]]

Take a moment to actually read `comparison["ollama_output"]` and `comparison["mistral_output"]` side by side, and fill in the `quality_notes` column by hand -- this is a qualitative judgement no automated metric captures well here.

A few things worth discussing once you've compared the outputs:

* **Hallucination risk on noisy OCR.** OCR'd 19th-century text is full of scanning artifacts (misrecognized characters, broken words, running headers mixed into body text). A model may confidently invent a place or date that isn't actually supported by the garbled text -- always spot-check against the original page image where possible.
* **Reproducibility.** We set `temperature=0` for the Mistral call and used `options={"temperature": 0}` for Ollama to make outputs as deterministic as possible, but this is *not* a guarantee: commercial APIs can still return slightly different outputs across calls for the same input, even at temperature 0, due to floating-point non-determinism, batching effects, or model updates on the provider's side.
* **Privacy.** Unpublished or restricted archival material sent to a third-party API leaves your infrastructure and is subject to that provider's data-retention policy. For sensitive collections, a local model (however much weaker) may be the only acceptable option.
* **Cost at scale.** A few cents per document looks negligible until you multiply by a collection of thousands of documents. Estimating cost on a small sample *before* committing to processing a full archive, as we just did, is good practice.

## Wrap-up: local vs. frontier

| | Local (Ollama) | Frontier API (Mistral/OpenAI) |
|---|---|---|
| **Cost** | Free after download (compute/electricity only) | Per-token, scales with usage |
| **Privacy** | Data never leaves your machine | Data sent to a third party, subject to their retention policy |
| **Output quality** | Good for small/fast models, generally behind frontier models | State of the art, especially on reasoning-heavy tasks |
| **Reproducibility** | More controllable (you own the weights and runtime) | Best-effort determinism even at `temperature=0` |
| **Offline capability** | Fully offline | Requires network access |

## Questions

## Why RAG

Even the best generative model only knows what was in its training data, up to its training cutoff, and it has no way of citing where a specific fact came from. **Retrieval-Augmented Generation (RAG)** addresses this by pairing the generator with a **retriever**: before answering, the system searches an external, curated collection of documents for passages relevant to the query, and feeds those passages to the model as extra context.

<img src="figures/rag.png" width="600px">

*[Source](https://www.trantorinc.com/blog/what-is-rag-retrieval-augmented-generation).*

The pipeline has four steps:

1. **Query encoding** -- the user's question is turned into a vector using an embedding model (the same kind of embedding we produced with `ollama.embed` above).
2. **Retrieval** -- that vector is compared against pre-computed embeddings of the document collection (usually stored in a vector database), and the top-k most similar passages are returned.
3. **Augmentation** -- those passages are inserted into the prompt alongside the original question.
4. **Generation** -- the generative model answers, conditioned on both its own pre-trained knowledge and the retrieved passages.

RAG is especially relevant for the Humanities, GLAM, and documentary fields: question-answering over a specific archive or reading list, fact-checking, and specialized conversational agents over a fixed knowledge base are all natural fits. Its main failure modes are retrieval quality (a bad retriever means the model answers with irrelevant context), retrieval speed at scale, and how to fuse several retrieved passages coherently -- all worth keeping in mind as you build the from-scratch version below.

## RAG from scratch

Before looking at the packaged `rag_app/` application, it's worth building the mechanics by hand once: extract text from the same six PDFs used by `rag_app/`, chunk it, embed each chunk locally with Ollama (no API cost), retrieve the most similar chunks to a query with plain cosine similarity (no vector database), and stuff them into a prompt.

We assume this notebook is run from the repository root, so the PDFs are at `rag_app/data/`.

In [ ]:
from pypdf import PdfReader
import pathlib

pdf_dir = pathlib.Path("rag_app/data")
pdf_paths = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdf_paths)} PDFs: {[p.name for p in pdf_paths]}")

def extract_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

documents = {p.name: extract_text(p) for p in pdf_paths}
print("Characters extracted from", pdf_paths[0].name, ":", len(documents[pdf_paths[0].name]))

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    """Fixed-size character chunking with overlap between consecutive chunks."""
    assert overlap < chunk_size, "overlap must be smaller than chunk_size, or chunking never advances"
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []       # list of chunk strings
chunk_sources = []    # parallel list of source filenames

for filename, text in documents.items():
    for chunk in chunk_text(text):
        all_chunks.append(chunk)
        chunk_sources.append(filename)

print(f"Total chunks: {len(all_chunks)}")

In [ ]:
import numpy as np

def embed_chunks(chunks, model="nomic-embed-text", batch_size=32):
    vectors = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        response = ollama.embed(model=model, input=batch)
        vectors.extend(response["embeddings"])
    return np.array(vectors)

chunk_embeddings = embed_chunks(all_chunks)
print("Embeddings shape:", chunk_embeddings.shape)

In [ ]:
def cosine_similarity(query_vec, matrix):
    query_norm = query_vec / np.linalg.norm(query_vec)
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix_norm @ query_norm

def retrieve(query, k=3, model="nomic-embed-text"):
    query_vec = np.array(ollama.embed(model=model, input=query)["embeddings"][0])
    similarities = cosine_similarity(query_vec, chunk_embeddings)
    top_k_idx = np.argsort(similarities)[::-1][:k]
    return [(all_chunks[i], chunk_sources[i], similarities[i]) for i in top_k_idx]

In [ ]:
def answer_with_rag(query, k=3, use_mistral=True):
    retrieved = retrieve(query, k=k)
    context = "\n\n---\n\n".join(f"[{source}]\n{chunk}" for chunk, source, _ in retrieved)

    rag_prompt = f'''Answer the question using ONLY the context below. If the context does not contain \
the answer, say you don't know -- do not use outside knowledge.

CONTEXT:
{context}

QUESTION: {query}
'''

    if use_mistral:
        response = with_backoff(lambda: client.chat.complete(
            model=MISTRAL_MODEL,
            max_tokens=512,
            messages=[{"role": "user", "content": rag_prompt}],
        ))
        answer = response.choices[0].message.content
    else:
        response = ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": rag_prompt}])
        answer = response["message"]["content"]

    print("ANSWER:\n", answer)
    print("\nRETRIEVED FROM:", [source for _, source, _ in retrieved])
    return answer, retrieved

_ = answer_with_rag("What are the main challenges of using machine learning on historical document images?")

# The call above can come back with "I don't know" -- but the answer does exist in the corpus:
# 3.pdf's "Final Remarks" section explicitly names data scarcity, dataset fragmentation, and
# reduced model explainability as the main challenges. This is a *retrieval* failure, not a
# missing-content one: at k=3 the top chunks simply didn't include that section.
# Retrieval runs locally via Ollama and costs nothing, so we can verify that without a second
# paid call -- widen k and look at what comes back:
question = "What are the main challenges of using machine learning on historical document images?"
for chunk, source, score in retrieve(question, k=8):
    print(f"[{source}] score={score:.3f} | {chunk[:100]}...")

Now try a question the six papers almost certainly don't cover -- for example, something about a completely unrelated domain. Retrieval will still return the *closest available* chunks (cosine similarity always returns a top-k, however weak the match), and the model has to decide what to do with clearly irrelevant context.

In [ ]:
_ = answer_with_rag("What is the recommended tire pressure for a 2020 Toyota Corolla?")

Depending on the prompt's wording, the model will typically either (a) correctly say the context doesn't answer the question, or (b) fall back on its own pre-trained knowledge and answer anyway -- which defeats the point of grounding the answer in your document collection. This is exactly the trade-off the `rag_app/app.py` system prompt makes explicit: it instructs the model to fall back on general knowledge but to flag it ("I am not entirely sure about this but ..."), rather than silently mixing retrieved and non-retrieved knowledge.

## Knobs: chunk size, overlap, and k

Three parameters control what ends up in the model's context: **chunk size** (how much text per chunk -- too small loses context, too large dilutes relevance and wastes tokens), **overlap** (how much consecutive chunks share, to avoid cutting a relevant sentence exactly at a chunk boundary), and **k** (how many chunks are retrieved -- too few may miss the answer, too many adds noise and cost). The embedding model and similarity metric (cosine here; other options include dot product or Euclidean distance) are two further knobs we haven't varied here, but which also affect retrieval quality.

In [ ]:
# Re-chunk with a different size/overlap and see how the retrieved passages change
small_chunks, small_sources = [], []
for filename, text in documents.items():
    for chunk in chunk_text(text, chunk_size=300, overlap=50):
        small_chunks.append(chunk)
        small_sources.append(filename)

small_embeddings = embed_chunks(small_chunks)

def retrieve_from(query, chunks, sources, embeddings, k=3):
    query_vec = np.array(ollama.embed(model="nomic-embed-text", input=query)["embeddings"][0])
    similarities = cosine_similarity(query_vec, embeddings)
    top_k_idx = np.argsort(similarities)[::-1][:k]
    return [(chunks[i], sources[i], similarities[i]) for i in top_k_idx]

query = "What are the main challenges of using machine learning on historical document images?"
for label, (c, s, e, k) in {
    "chunk_size=1000, overlap=200, k=3": (all_chunks, chunk_sources, chunk_embeddings, 3),
    "chunk_size=300, overlap=50, k=5": (small_chunks, small_sources, small_embeddings, 5),
}.items():
    print(f"--- {label} ---")
    for chunk, source, score in retrieve_from(query, c, s, e, k=k):
        print(f"[{source}] score={score:.3f} | {chunk[:120]}...")
    print()

Smaller chunks tend to retrieve more precise passages but lose surrounding context; larger chunks give more context per hit but at higher token cost and a higher chance of including irrelevant material alongside the relevant sentence. There is no universally correct setting -- it depends on the structure of your documents and the kind of questions you expect.

## From notebook to app: `rag_app/`

The from-scratch version above makes the mechanics explicit, but a real application needs a UI, session management, and a proper vector index. `rag_app/app.py` packages the same ideas using **Chainlit** (chat UI and session lifecycle) and **LlamaIndex** (indexing and retrieval):

* **`auth_callback`** -- a simple username/password gate, reading `CHAINLIT_USERNAME` and `CHAINLIT_PASSWORD` from the environment.
* **`@cl.on_chat_start`** -- runs once per new chat session. It loads every file under `rag_app/data/` with `SimpleDirectoryReader`, builds a `VectorStoreIndex` over them, and wraps the index in a **context-mode chat engine** (`chat_mode="context"`) with `similarity_top_k=3` and a hardcoded `system_prompt` that explicitly enumerates the six papers in the knowledge base (the same reading-list PDFs we used above). Chunking is configured via `Settings.chunk_size = 1024` and `Settings.chunk_overlap = 32` -- the same two knobs we varied by hand a moment ago.
* **`@cl.on_message`** -- streams the model's answer token by token (`chat_engine.stream_chat`), then sends a follow-up "Sources" message listing the distinct source PDFs behind that answer, rendered inline in the UI via `cl.Pdf`.

By default the app uses **OpenAI** (`gpt-4-turbo` for generation, `text-embedding-3-small` for embeddings). The commented-out lines just below --

```python
#Settings.llm = Ollama(model="llama3.1", request_timeout=360.0, temperature=temperature, max_tokens=max_tokens, streaming=streaming)
#Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")
```

-- are the "swap to local" mechanism: comment out the two OpenAI `Settings` lines above them, uncomment these two, and the same application runs entirely on your machine via Ollama, at the cost of some quality and speed.

**Running it.** From inside `rag_app/`:

```
chainlit run app.py
```

with `CHAINLIT_USERNAME`, `CHAINLIT_PASSWORD`, and `OPENAI_API_KEY` set as environment variables (again, via a `.env` file -- never hardcoded).

## Exercises

From `rag_app/README.md`:

1. Individuate your own documents of interest and use the app on them (swap out the six PDFs in `rag_app/data/`).
2. Add metadata to the documents to facilitate their usage by the LLM, and experiment with prompting (rewrite `prompt_text` in `app.py`).
3. Add customizations to the interface, such as the possibility to upload a new file, and to tweak the LLM's parameters.
4. Change the vector database and seek something more performant than the default in-memory `VectorStoreIndex`.
5. Try and compare different LLMs, including locally using [Ollama](https://ollama.com/) (the commented-out lines in `app.py` are your starting point).

Two additional exercises for this course:

6. **Swap the LLM backend in `app.py` from OpenAI to Mistral.** This requires adding a Mistral LlamaIndex integration -- `llama-index-llms-mistralai` and `llama-index-embeddings-mistralai` -- which are **not currently listed** in `requirements.txt`, so you'll need to add them yourself (`pip install llama-index-llms-mistralai llama-index-embeddings-mistralai`), then replace the `OpenAI(...)` / `OpenAIEmbedding(...)` `Settings` with `MistralAI(...)` / `MistralAIEmbedding(...)`, using `model="mistral-large-latest"` and `model_name="mistral-embed"` respectively.
7. **Measure retrieval quality.** Write 10 question/expected-source-PDF pairs about the six papers, run each question through the app (or through the from-scratch retriever above), and measure the retrieval **hit-rate**: did the correct source PDF appear among the top-k retrieved chunks?

## Questions & wrap-up

That's it for the course notebooks. As a follow-up, consider proposing a dedicated workshop to go deeper into any of these topics -- fine-tuning open-weight models, building a production RAG pipeline, or evaluating LLM outputs systematically -- as mentioned in the course program.